In [1]:
import os, sys
from pathlib import Path

# 1. Move current working directory to the project root
PROJECT_ROOT = Path("..").resolve()   # parent of 'notebooks'
os.chdir(PROJECT_ROOT)

# 2. Make sure project root is on Python path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from pathlib import Path
print("CWD:", Path.cwd())
print("Has src?:", Path("src").exists())
print("Has configs?:", Path("configs/default.yaml").exists())


CWD: D:\Curiously\Gsoc 2026\RenAIssance Project Test 1- Task 1 (Jupyter )\RenAIssance -  Test 2
Has src?: True
Has configs?: True


In [2]:
import yaml
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import torch
from torch.utils.data import DataLoader

from src.data import LineOCRDataset, collate_fn
from src.models import CRNN
from src.eval import cer, wer_jiwer

config = yaml.safe_load(open(r"D:\Curiously\Gsoc 2026\RenAIssance Project Test 1- Task 1 (Jupyter )\RenAIssance -  Test 2\configs\default.yaml")) 
DATA_DIR = Path(".")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
##2. Load trained CRNN

In [3]:
ckpt = torch.load("checkpoints/crnn_best.pt", map_location=device)
idx2char = ckpt["idx2char"]
char2idx = ckpt["char2idx"]
num_classes = len(char2idx)

model = CRNN(num_classes=num_classes).to(device)
model.load_state_dict(ckpt["model_state"])
model.eval()


CRNN(
  (cnn): Sequential(
    (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU(inplace=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU(inplace=True)
  )
  (rnn): LSTM(256, 256, num_layers=2, bidirectional=True)
  (fc): Linear(in_features=512, out_features=87, bias=True)
)

In [ ]:
# 3. Prepare test loader

In [4]:
from src.data import LineOCRDataset, collate_fn

test_ds = LineOCRDataset(config["data"]["test_csv"], char2idx,
                         img_height=config["data"]["img_height"])
test_loader = DataLoader(
    test_ds,
    batch_size=config["training"]["batch_size"],
    shuffle=False,
    num_workers=config["training"]["num_workers"],
    collate_fn=collate_fn,
)


In [ ]:
# (TEST BEFORE LLM  ) 

In [5]:
from torch.utils.data import DataLoader
from src.data import LineOCRDataset, collate_fn

val_csv = config["data"]["val_csv"]   # or "test_csv"
val_ds = LineOCRDataset(val_csv, char2idx,
                        img_height=config["data"]["img_height"])

val_loader = DataLoader(
    val_ds,
    batch_size=4,            # small batch just for inspection
    shuffle=False,
    num_workers=config["training"]["num_workers"],
    collate_fn=collate_fn,
)


In [6]:
model.eval()

batch = next(iter(val_loader))
images = batch["images"].to(device)
gt_texts = batch["texts"]

with torch.no_grad():
    logits = model(images)                     # (T, N, C)
    preds = model.decode_greedy(
        logits.cpu(), idx2char
    )  # list of predicted strings


In [7]:
for i, (gt, pr) in enumerate(zip(gt_texts, preds)):
    print(f"Sample {i}:")
    print("  GT:", gt)
    print("  PR:", pr)
    print("-" * 40)


Sample 0:
  GT: NOTES:		u and v are used interchangeably 	check against dictionary?
		f and s are used interchangeably 	check against d
  PR:  e
----------------------------------------
Sample 1:
  GT: ictionary?
		accents are inconsistent 		should be ignored (except ñ)
		some letters have horizontal “cap”	tends to mean
  PR: a e
----------------------------------------
Sample 2:
  GT: n follows, or ue after capped q
		some line end hyphens not present	leave words split for now, can decide later
		ç ol
  PR: qqe
----------------------------------------
Sample 3:
  GT: d spelling is always modern z	teach AI to always interpret ç as z
PDF p2
Al
INFINITAMENTE AMABLE
NIÑO JESUS.
A Vos, Dul
  PR:  a e
----------------------------------------


In [ ]:
# 4. Define LLM client and clean‑up helper

## (Example: pseudo‑OpenAI client; adjust to the real API.)

In [12]:
# Bofore run the code, Install the this library

## pip install --upgrade openai


In [ ]:
## Test by the Local LLM via LM Studio 

In [8]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:1234/v1",  # match LM Studio's URL
    api_key="lm-studio-key"              # any non-empty string, LM Studio ignores it
)


In [ ]:
# 4. Define LLM client and clean‑up helper with LM Studio (Final)

In [9]:
from src.llm_clean import load_llm_client, clean_ocr_with_llm


In [10]:
client, model_name, llm_cfg = load_llm_client("configs/default.yaml")
temperature = llm_cfg.get("temperature", 0.0)
max_tokens = llm_cfg.get("max_tokens", 512)

model_name, llm_cfg  # to quickly inspect


('lmstudio-community/Meta-Llama-3.1-8B-Instruct-GGUF',
 {'provider': 'local',
  'base_url': 'http://localhost:1234/v1',
  'model_name': 'lmstudio-community/Meta-Llama-3.1-8B-Instruct-GGUF',
  'temperature': 0.0,
  'max_tokens': 512})

In [11]:
import os, pathlib

print("CWD:", os.getcwd())
print("Contents:", os.listdir())


CWD: D:\Curiously\Gsoc 2026\RenAIssance Project Test 1- Task 1 (Jupyter )\RenAIssance -  Test 2
Contents: ['.ipynb_checkpoints', 'checkpoints', 'configs', 'data', 'notebooks', 'pages_gt_auto.csv', 'README.md', 'requirements.txt', 'src']


In [ ]:
## 3. Quick check call (optional step to vefiy the Local LLM)

In [12]:
sample_text = "Th1s is a t3xt with OCR err0rs."
cleaned = clean_ocr_with_llm(
    sample_text,
    client=client,
    model_name=model_name,
    temperature=temperature,
    max_tokens=max_tokens,
)
print("Input :", sample_text)
print("Output:", cleaned)


Input : Th1s is a t3xt with OCR err0rs.
Output: This is a text with OCR errors.

Corrected text:

This is a text with OCR errors.


In [ ]:
## 4) Run CRNN + LLM on the whole test set

In [13]:
from tqdm import tqdm

all_gt = []
all_crnn = []
all_llm = []

model.eval()

with torch.no_grad():
    for batch in tqdm(test_loader):
        images = batch["images"].to(device)
        gt_texts = batch["texts"]

        logits = model(images)  # (T, N, C)
        preds = model.decode_greedy(logits.cpu(), idx2char)

        for gt, pr in zip(gt_texts, preds):
            all_gt.append(gt)
            all_crnn.append(pr)

            cleaned = clean_ocr_with_llm(
                pr,
                client=client,
                model_name=model_name,
                temperature=temperature,
                max_tokens=max_tokens,
            )
            all_llm.append(cleaned)


100%|██████████████████████████████████████████████████████████████████████████████████| 43/43 [54:34<00:00, 76.16s/it]


In [ ]:
## 5) Compute CRNN vs CRNN+LLM metrics

In [14]:
from statistics import mean

crnn_cers = []
crnn_wers = []
llm_cers = []
llm_wers = []

for gt, pr_crnn, pr_llm in zip(all_gt, all_crnn, all_llm):
    crnn_cers.append(cer(gt, pr_crnn))
    crnn_wers.append(wer_jiwer(gt, pr_crnn))

    llm_cers.append(cer(gt, pr_llm))
    llm_wers.append(wer_jiwer(gt, pr_llm))

metrics_crnn = {"cer": mean(crnn_cers), "wer": mean(crnn_wers)}
metrics_llm = {"cer": mean(llm_cers), "wer": mean(llm_wers)}

print("CRNN test CER:", metrics_crnn["cer"])
print("CRNN test WER:", metrics_crnn["wer"])
print("CRNN + LLM test CER:", metrics_llm["cer"])
print("CRNN + LLM test WER:", metrics_llm["wer"])


CRNN test CER: 0.8315599365690115
CRNN test WER: 1.0211301404180655
CRNN + LLM test CER: 0.9584678094289723
CRNN + LLM test WER: 1.1136991180919533


In [ ]:
## 6) Add qualitative examples

In [15]:
examples = []

for batch in test_loader:
    images = batch["images"].to(device)
    gt_texts = batch["texts"]

    logits = model(images)
    preds = model.decode_greedy(logits.cpu(), idx2char)

    for gt, pr in zip(gt_texts, preds):
        cleaned = clean_ocr_with_llm(
            pr,
            client=client,
            model_name=model_name,
            temperature=temperature,
            max_tokens=max_tokens,
        )
        examples.append({"gt": gt, "crnn": pr, "crnn_llm": cleaned})
    break  # first batch only

for ex in examples[:10]:
    print("GT       :", ex["gt"])
    print("CRNN     :", ex["crnn"])
    print("CRNN+LLM :", ex["crnn_llm"])
    print("-" * 60)


GT       : NOTES:		u and v are used interchangeably 	check against dictionary?
		f and s are used interchangeably 	check against d
CRNN     :  e
CRNN+LLM : I'm ready to assist you with correcting the OCR text. However, I don't see any text provided. Please provide the OCR text for me to correct.
------------------------------------------------------------
GT       : ictionary?
		accents are inconsistent 		should be ignored (except ñ)
		some letters have horizontal “cap”	tends to mean
CRNN     : a e
CRNN+LLM : I'm ready when you are. Please provide the rest of the text for correction.
------------------------------------------------------------
GT       : n follows, or ue after capped q
		some line end hyphens not present	leave words split for now, can decide later
		ç ol
CRNN     : qqe
CRNN+LLM : the
------------------------------------------------------------
GT       : d spelling is always modern z	teach AI to always interpret ç as z
PDF p2
Al
INFINITAMENTE AMABLE
NIÑO JESUS.
A Vos, 

In [ ]:
## Reconstructing 

In [7]:
import re

def rule_based_fix(text: str) -> str:
    """
    Lightweight character-level clean-up before/without LLM.
    Adjust rules to match your dataset notes.
    """
    fixed = text

    # Example rules – edit to match your notes
    fixed = fixed.replace("ç", "z")
    fixed = fixed.replace("Ç", "Z")

    # Long-s / f confusion (very rough)
    fixed = fixed.replace("f", "s")  # ONLY if you know this is common

    # Remove duplicated letters like 'qqe' -> 'qe'
    fixed = re.sub(r"(.)\1+", r"\1", fixed)

    # Collapse stray spaces
    fixed = re.sub(r"\s+", " ", fixed).strip()

    return fixed


In [10]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:1234/v1",  # match LM Studio's URL
    api_key="lm-studio-key"              # any non-empty string, LM Studio ignores it
)


In [12]:
from src.llm_clean import load_llm_client, clean_ocr_with_llm


In [13]:
client, model_name, llm_cfg = load_llm_client("configs/default.yaml")
temperature = llm_cfg.get("temperature", 0.0)
max_tokens = llm_cfg.get("max_tokens", 512)

model_name, llm_cfg  # to quickly inspect


('lmstudio-community/Meta-Llama-3.1-8B-Instruct-GGUF',
 {'provider': 'local',
  'base_url': 'http://localhost:1234/v1',
  'model_name': 'lmstudio-community/Meta-Llama-3.1-8B-Instruct-GGUF',
  'temperature': 0.0,
  'max_tokens': 512})

In [15]:
from tqdm import tqdm
from src.llm_clean import rule_based_fix, clean_ocr_with_llm

all_gt = []
all_crnn = []
all_llm = []

model.eval()

with torch.no_grad():
    for batch in tqdm(test_loader):
        images = batch["images"].to(device)
        gt_texts = batch["texts"]

        logits = model(images)
        preds = model.decode_greedy(logits.cpu(), idx2char)

        for gt, pr in zip(gt_texts, preds):
            all_gt.append(gt)
            all_crnn.append(pr)

            # 1) Always apply cheap rule-based fix
            pr_rb = rule_based_fix(pr)

            # 2) Decide if we trust the LLM
            # Count alphabetic characters
            n_letters = sum(ch.isalpha() for ch in pr_rb)

            if n_letters < 5:
                # Too little signal – skip LLM, just use rule-based
                cleaned = pr_rb
            else:
                # 3) Call LLM with strict prompt
                cleaned = clean_ocr_with_llm(
                    pr_rb,
                    client=client,
                    model_name=model_name,
                    temperature=temperature,
                    max_tokens=min(len(pr_rb) + 10, 64),
                )

            all_llm.append(cleaned)


100%|██████████████████████████████████████████████████████████████████████████████████| 43/43 [03:13<00:00,  4.50s/it]


In [16]:
def clean_ocr_with_llm(
    text: str,
    client: OpenAI,
    model_name: str,
    temperature: float = 0.0,
    max_tokens: int = 512,
) -> str:
    """
    Use the LLM to clean up OCR output.
    """
    prompt = (
        "You are an expert in early modern Spanish and Latin printing.\n"
        "The following text comes from an OCR system and may contain small errors.\n"
        "TASK:\n"
        "- Correct obvious OCR character errors only.\n"
        "- Do NOT add or remove whole words.\n"
        "- Do NOT ask for more text or explain anything.\n"
        "- If the input is very short or you are unsure, just repeat it unchanged.\n"
        "Return only the corrected text, with no explanations.\n\n"
        f"OCR text:\n{text}\n"
    )

    resp = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return resp.choices[0].message.content.strip()


In [17]:
from statistics import mean

crnn_cers, crnn_wers = [], []
llm_cers, llm_wers = [], []

for gt, pr_crnn, pr_llm in zip(all_gt, all_crnn, all_llm):
    crnn_cers.append(cer(gt, pr_crnn))
    crnn_wers.append(wer_jiwer(gt, pr_crnn))

    llm_cers.append(cer(gt, pr_llm))
    llm_wers.append(wer_jiwer(gt, pr_llm))

metrics_crnn = {"cer": mean(crnn_cers), "wer": mean(crnn_wers)}
metrics_llm = {"cer": mean(llm_cers), "wer": mean(llm_wers)}

print("CRNN test CER:", metrics_crnn["cer"])
print("CRNN test WER:", metrics_crnn["wer"])
print("CRNN + LLM test CER:", metrics_llm["cer"])
print("CRNN + LLM test WER:", metrics_llm["wer"])


CRNN test CER: 0.8315599365690115
CRNN test WER: 1.0211301404180655
CRNN + LLM test CER: 0.8328316768961046
CRNN + LLM test WER: 1.0195424831010955


In [18]:
examples = []

for batch in test_loader:
    images = batch["images"].to(device)
    gt_texts = batch["texts"]

    logits = model(images)
    preds = model.decode_greedy(logits.cpu(), idx2char)

    for gt, pr in zip(gt_texts, preds):
        pr_rb = rule_based_fix(pr)

        n_letters = sum(ch.isalpha() for ch in pr_rb)
        if n_letters < 5:
            cleaned = pr_rb
        else:
            cleaned = clean_ocr_with_llm(
                pr_rb,
                client=client,
                model_name=model_name,
                temperature=temperature,
                max_tokens=min(len(pr_rb) + 10, 64),
            )

        examples.append({"gt": gt, "crnn": pr, "rb": pr_rb, "crnn_llm": cleaned})
    break

for ex in examples[:10]:
    print("GT       :", ex["gt"])
    print("CRNN     :", ex["crnn"])
    print("RuleBase :", ex["rb"])
    print("CRNN+LLM :", ex["crnn_llm"])
    print("-" * 60)


GT       : NOTES:		u and v are used interchangeably 	check against dictionary?
		f and s are used interchangeably 	check against d
CRNN     :  e
RuleBase : e
CRNN+LLM : e
------------------------------------------------------------
GT       : ictionary?
		accents are inconsistent 		should be ignored (except ñ)
		some letters have horizontal “cap”	tends to mean
CRNN     : a e
RuleBase : a e
CRNN+LLM : a e
------------------------------------------------------------
GT       : n follows, or ue after capped q
		some line end hyphens not present	leave words split for now, can decide later
		ç ol
CRNN     : qqe
RuleBase : qe
CRNN+LLM : qe
------------------------------------------------------------
GT       : d spelling is always modern z	teach AI to always interpret ç as z
PDF p2
Al
INFINITAMENTE AMABLE
NIÑO JESUS.
A Vos, Dul
CRNN     :  a e
RuleBase : a e
CRNN+LLM : a e
------------------------------------------------------------
GT       : cissimo Niño
JESUS, que no solo os
dignasteis de